# Prepare and stage the fraud data

[Guide](README.md) · [00: prepare data](00_prepare_data.ipynb) · [01: experiment](01_experiment.ipynb) · [02: results](02_results.ipynb) · [03: predictor](03_predictor.ipynb)

Read the [prerequisites and Connection setup](README.md#before-opening-the-notebooks) and fill in [`.env.example`](.env.example) as `.env` before uploading. This notebook first explores local data; only the **Upload** cell contacts S3. All seven repository features are retained.

In [ ]:
from pathlib import Path
requirements = Path('requirements.txt') if Path('workflow.py').exists() else Path('automl/requirements.txt')
%pip install -r {requirements}

Restart the kernel if installation requests it, then continue below.

In [2]:
from pathlib import Path
import sys

# Works when Jupyter starts in either the repo root or automl/.
automl_dir = Path.cwd() if Path('workflow.py').exists() else Path.cwd() / 'automl'
sys.path.insert(0, str(automl_dir.resolve()))
import pandas as pd
from IPython.display import display, Markdown, FileLink
import workflow as wf

settings = wf.settings()  # Never display this mapping: it may contain credentials.

## Explore the splits and class imbalance

The label is `fraud` (1 = fraud, 0 = not fraud). Names come directly from the CSV headers, including underscores. No features are scaled or discarded.

In [3]:
frames = wf.load_data()
display(frames['train'].head())
summary = pd.DataFrame(wf.data_summary(frames))
display(summary.style.format({'fraud_rate': '{:.2%}'}))
print('Features:', ', '.join(wf.FEATURES))
print('A majority-class-only classifier would have accuracy:',
      f"{1 - frames['test']['fraud'].mean():.2%}")

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,75.744071,0.526379,1.178682,1.0,0.0,0.0,1.0,0
1,3.607799,0.687956,0.081721,1.0,1.0,0.0,1.0,0
2,14.303444,8.439479,0.316733,1.0,1.0,0.0,1.0,0
3,113.614272,0.527061,1.748520,1.0,0.0,0.0,1.0,1
4,96.346684,0.448277,1.373413,1.0,0.0,0.0,1.0,0


,split,rows,fraud_rows,fraud_rate
0,train,600000,52535,8.76%
1,validate,200000,17485,8.74%
2,test,200000,17383,8.69%


Features: distance_from_home, distance_from_last_transaction, ratio_to_median_purchase_price, repeat_retailer, used_chip, used_pin_number, online_order
A majority-class-only classifier would have accuracy: 91.31%


## Choose the training data

`MERGE_VALIDATION=true` combines the original train and validation splits, since AutoML manages its own training splits. Set it to `false` before starting the kernel to use only `train.csv`. The external `test.csv` is never added to training. The managed pipeline uses it for evaluation and candidate ranking; see the [evaluation caveat](README.md#interpreting-the-results).

In [4]:
print('Merge validation:', settings.get('MERGE_VALIDATION', 'true'))
print('Label:', wf.LABEL, '| Binary classification | Positive class: 1')

Merge validation: true
Label: fraud | Binary classification | Positive class: 1


## Upload the prepared CSVs to S3

Run this cell when ready to write data. Each execution creates a unique input prefix, verifies uploaded sizes and saves a manifest locally and in S3. It does not modify the repository CSVs. Existing buckets and Connections are required.

In [6]:
s3 = wf.s3_client()
manifest = wf.stage_data(frames, s3)
display(pd.DataFrame(manifest['datasets']).T[['uri', 'rows', 'fraud_rate', 'sha256']])
print('Saved state:', wf.OUTPUTS / 'data_manifest.json')

,uri,rows,fraud_rate,sha256
train,s3://ai-eng-cracow/fraud-detection/automl/data...,800000,0.087525,84d4b615956c2391739a29a56df3f914749bebaa8544e9...
test,s3://ai-eng-cracow/fraud-detection/automl/data...,200000,0.086915,d05f8f71891618474de701f15f45b23bdfb722c9087e12...


Saved state: /Users/mszewczy/projects/fraud-detection/automl/outputs/data_manifest.json


Next: [01_experiment.ipynb](01_experiment.ipynb)